# DE-GIPCA: Differential Evolution on Grassmannian

This notebook demonstrates the **Differential Evolution GIPCA** estimator that optimizes directly on the Grassmannian manifold.

## Model

The GIPCA model with macro factors:
$$r_t = Z_t W (f^0_t + \Delta \mu_t) + \epsilon_t$$

Where:
- $W \in \text{Gr}(m, k)$: Characteristic-to-loading map (Grassmannian)
- $\Delta \in \mathbb{R}^{k \times l}$: Macro-to-factor map
- $f^0_t \in \mathbb{R}^k$: Residual factor component
- $\mu_t \in \mathbb{R}^l$: Macro variables

## Estimation Method

The estimator uses a **profiled objective**:
1. For a given $W$, solve for $\Delta$ and $f^0$ in closed form
2. Use Differential Evolution to optimize $W$ on the Grassmannian

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, '../src')

from de_gipca import GrassmannGIPCAEstimator, generate_gipca_data

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Generate Simulated Data

We use `generate_gipca_data()` which creates:
- True $W^*$ on the Grassmannian
- True $\Delta^*$ (macro-to-factor map)
- Macro series $\mu_t$ with AR(1) dynamics
- Factor residuals $f^0_t$ with AR(1) dynamics
- Returns with heteroskedastic noise

In [ ]:
# Simulation parameters
seed = 42
T = 50           # time periods (keep small for DE speed)
N = 80           # assets
m = 10           # characteristics
k = 3            # factors
num_macro = 4    # macro variables

print("Simulation Parameters:")
print(f"  T (time periods):      {T}")
print(f"  N (assets):            {N}")
print(f"  m (characteristics):   {m}")
print(f"  k (factors):           {k}")
print(f"  l (macro variables):   {num_macro}")

In [ ]:
# Generate data
data, truth = generate_gipca_data(
    T=T,
    N=N,
    m=m,
    k=k,
    num_macro=num_macro,
    seed=seed,
    include_intercept=False,  # No intercept for simplicity
    sigma_eps_base=0.3,       # Noise level
    missing_prob=0.0,         # No missing values
)

rets, Z, mu = data

print(f"\nData shapes:")
print(f"  rets: {rets.shape}")
print(f"  Z:    {Z.shape}")
print(f"  mu:   {mu.shape}")

In [ ]:
# True parameters
W_star = truth['W_star']       # m x k
Delta_star = truth['Delta_star']  # k x l
f0_true = truth['f0']          # T x k
f_full_true = truth['f_full']  # T x k (f0 + Delta @ mu)

print("True Parameters:")
print(f"  W_star shape:     {W_star.shape}")
print(f"  Delta_star shape: {Delta_star.shape}")
print(f"  f0_true shape:    {f0_true.shape}")
print(f"  f_full shape:     {f_full_true.shape}")

# Verify W_star is orthonormal
print(f"\nW_star orthonormality (W'W):")
print(np.round(W_star.T @ W_star, 4))

In [ ]:
# Plot macro variables
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
for l in range(num_macro):
    ax.plot(mu[:, l], label=f'Macro {l+1}')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.set_title('Macro Variables $\\mu_t$')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
for kk in range(k):
    ax.plot(f_full_true[:, kk], label=f'Factor {kk+1}')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.set_title('True Full Factors $f_t = f^0_t + \\Delta \\mu_t$')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Fit DE-GIPCA Model

The `GrassmannGIPCAEstimator` uses Differential Evolution to optimize $W$ on the Grassmannian manifold. For each candidate $W$, it solves for $\Delta$ in closed form using the profiled objective.

In [ ]:
# Initialize estimator
estimator = GrassmannGIPCAEstimator(
    num_assets=N,
    num_fact=k,
    num_charact=m,
    num_macro=num_macro,
    win_len=T
)

print(f"Estimator initialized:")
print(f"  Grassmann dimension: Gr({m}, {k})")
print(f"  Optimization dim:    {estimator.dim}")
print(f"  Population size:     {5 * estimator.dim}")

In [ ]:
# Fit the model (this may take a few minutes)
print("Fitting DE-GIPCA model...")
print("="*60)

max_gen = 500  # Maximum generations for DE

W_est, Delta_est, f0_est, history = estimator.fit(data, max_gen=max_gen, ridge=1e-6)

In [ ]:
# Compute estimated full factors
f_full_est = f0_est + (Delta_est @ mu.T).T  # T x k

print(f"\nEstimated Parameters:")
print(f"  W_est shape:     {W_est.shape}")
print(f"  Delta_est shape: {Delta_est.shape}")
print(f"  f0_est shape:    {f0_est.shape}")

## 3. Convergence Analysis

In [ ]:
# Plot DE convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(history, 'b-', linewidth=1.5)
ax.set_xlabel('Generation')
ax.set_ylabel('Objective')
ax.set_title('DE Convergence (Profiled Objective)')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.semilogy(history, 'b-', linewidth=1.5)
ax.set_xlabel('Generation')
ax.set_ylabel('Objective (log scale)')
ax.set_title('DE Convergence (Log Scale)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial objective: {history[0]:.6f}")
print(f"Final objective:   {history[-1]:.6f}")
print(f"Reduction:         {(1 - history[-1]/history[0])*100:.2f}%")

## 4. Evaluate Results

In [ ]:
# Grassmann distance between true and estimated W
Q1, _ = np.linalg.qr(W_star)
Q2, _ = np.linalg.qr(W_est)
_, s, _ = np.linalg.svd(Q1.T @ Q2)
principal_angles = np.arccos(np.clip(s, -1, 1))
grassmann_dist = np.linalg.norm(principal_angles)

print("Subspace Comparison (True W vs Estimated W):")
print(f"  Principal angles (degrees): {np.round(np.degrees(principal_angles), 2)}")
print(f"  Grassmann distance:         {grassmann_dist:.4f}")

In [ ]:
# Compute R-squared
preds = np.zeros((T, N))
for t in range(T):
    Lambda_t = Z[t] @ W_est
    preds[t, :] = Lambda_t @ f_full_est[t, :]

ss_res = np.sum((rets - preds) ** 2)
ss_tot = np.sum((rets - np.mean(rets)) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"\nModel Fit:")
print(f"  Overall R-squared: {r2:.4f}")

In [ ]:
# Factor correlations
factor_corr = np.corrcoef(f_full_est.T, f_full_true.T)[:k, k:]

print("\nFactor Correlations (Estimated vs True):")
print(np.round(np.abs(factor_corr), 4))

# Best matching correlations
best_corrs = []
for kk in range(k):
    best_corr = np.max(np.abs(factor_corr[kk, :]))
    best_corrs.append(best_corr)
print(f"\nBest matching |correlation| per estimated factor: {np.round(best_corrs, 4)}")
print(f"Mean best correlation: {np.mean(best_corrs):.4f}")

In [ ]:
# Delta comparison
delta_error = np.linalg.norm(Delta_est - Delta_star, 'fro')
delta_rel_error = delta_error / np.linalg.norm(Delta_star, 'fro')

print("\nDelta Recovery:")
print(f"  Frobenius error:    {delta_error:.4f}")
print(f"  Relative error:     {delta_rel_error:.4f}")

## 5. Visualizations

In [ ]:
# Plot estimated vs true factors
fig, axes = plt.subplots(k, 1, figsize=(14, 3*k))
if k == 1:
    axes = [axes]

for kk in range(k):
    ax = axes[kk]
    
    # Find best matching true factor
    best_corr = 0
    best_idx = 0
    for j in range(k):
        corr = np.abs(np.corrcoef(f_full_est[:, kk], f_full_true[:, j])[0, 1])
        if corr > best_corr:
            best_corr = corr
            best_idx = j
    
    # Align sign
    sign = np.sign(np.corrcoef(f_full_est[:, kk], f_full_true[:, best_idx])[0, 1])
    
    ax.plot(f_full_true[:, best_idx], 'b-', linewidth=2, alpha=0.7, label=f'True Factor {best_idx+1}')
    ax.plot(sign * f_full_est[:, kk], 'r--', linewidth=2, alpha=0.7, label=f'Estimated Factor {kk+1}')
    
    ax.set_xlabel('Time')
    ax.set_ylabel('Factor Value')
    ax.set_title(f'Estimated Factor {kk+1} vs Best Match (|corr| = {best_corr:.3f})')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmaps: W and Delta
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# True W
ax = axes[0, 0]
im = ax.imshow(W_star, aspect='auto', cmap='RdBu_r')
ax.set_xlabel('Factors')
ax.set_ylabel('Characteristics')
ax.set_title('True W (Gamma)')
plt.colorbar(im, ax=ax)

# Estimated W
ax = axes[0, 1]
im = ax.imshow(W_est, aspect='auto', cmap='RdBu_r')
ax.set_xlabel('Factors')
ax.set_ylabel('Characteristics')
ax.set_title('Estimated W (Gamma)')
plt.colorbar(im, ax=ax)

# True Delta
ax = axes[1, 0]
im = ax.imshow(Delta_star, aspect='auto', cmap='RdBu_r')
ax.set_xlabel('Macro Variables')
ax.set_ylabel('Factors')
ax.set_title('True Delta')
plt.colorbar(im, ax=ax)

# Estimated Delta
ax = axes[1, 1]
im = ax.imshow(Delta_est, aspect='auto', cmap='RdBu_r')
ax.set_xlabel('Macro Variables')
ax.set_ylabel('Factors')
ax.set_title('Estimated Delta')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted returns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
ax = axes[0]
ax.scatter(rets.flatten(), preds.flatten(), alpha=0.1, s=1)
ax.plot([rets.min(), rets.max()], [rets.min(), rets.max()], 'r--', linewidth=2)
ax.set_xlabel('Actual Returns')
ax.set_ylabel('Predicted Returns')
ax.set_title(f'Actual vs Predicted (R² = {r2:.4f})')
ax.grid(True, alpha=0.3)

# R² over time
ax = axes[1]
r2_ts = np.zeros(T)
for t in range(T):
    ss_res_t = np.sum((rets[t, :] - preds[t, :]) ** 2)
    ss_tot_t = np.sum((rets[t, :] - np.mean(rets[t, :])) ** 2)
    r2_ts[t] = 1 - ss_res_t / ss_tot_t if ss_tot_t > 0 else 0

ax.plot(r2_ts, 'b-', linewidth=1.5)
ax.axhline(y=r2, color='r', linestyle='--', label=f'Overall R² = {r2:.4f}')
ax.axhline(y=np.mean(r2_ts), color='g', linestyle=':', label=f'Mean R² = {np.mean(r2_ts):.4f}')
ax.set_xlabel('Time')
ax.set_ylabel('R²')
ax.set_title('Cross-sectional R² over Time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Summary

In [ ]:
print("="*70)
print("DE-GIPCA SIMULATION RESULTS SUMMARY")
print("="*70)

print(f"\nData Dimensions:")
print(f"  T (time periods):      {T}")
print(f"  N (assets):            {N}")
print(f"  m (characteristics):   {m}")
print(f"  k (factors):           {k}")
print(f"  l (macro variables):   {num_macro}")

print(f"\nOptimization:")
print(f"  Method:                Differential Evolution on Gr({m},{k})")
print(f"  Generations:           {len(history)}")
print(f"  Initial objective:     {history[0]:.6f}")
print(f"  Final objective:       {history[-1]:.6f}")

print(f"\nModel Performance:")
print(f"  Overall R²:            {r2:.4f}")
print(f"  Mean per-period R²:    {np.mean(r2_ts):.4f}")

print(f"\nParameter Recovery:")
print(f"  Grassmann distance:    {grassmann_dist:.4f}")
print(f"  Principal angles (°):  {np.round(np.degrees(principal_angles), 2)}")
print(f"  Delta relative error:  {delta_rel_error:.4f}")

print(f"\nFactor Recovery:")
print(f"  Mean best |corr|:      {np.mean(best_corrs):.4f}")

print("="*70)

In [ ]:
# Display estimated parameters
print("\nEstimated W (Gamma):")
print(np.round(W_est, 4))

print("\nEstimated Delta:")
print(np.round(Delta_est, 4))

In [ ]:
# Compare with true parameters
print("\nTrue W (Gamma):")
print(np.round(W_star, 4))

print("\nTrue Delta:")
print(np.round(Delta_star, 4))